# Notebook B — Flan-T5 fine-tuning for question generation

This notebook **fine-tunes** `google/flan-t5-base` on the **training split** of the same learning-resources QG data used in Notebook A. Inputs follow the project convention: `generate question: <context with <hl> answer </hl>>`.

You will: reload the dataset and **the same train/validation/test manifest** as Notebook A (run Notebook A first, or this notebook recreates the split), **fine-tune** with **validation loss each epoch**, then score **validation** and **test** generations with BLEU / ROUGE-L / METEOR and plots, and **compare** test metrics to `baseline_metrics.json` from Notebook A.

## 1. Environment: paths and imports

**What this section does:** Locates `backend/`, adds it to `sys.path`, and imports evaluation utilities. Training uses `transformers`, `datasets`, and `torch` (GPU used when available).

In [1]:
import json
import random
import sys
from pathlib import Path


def find_backend():
    cwd = Path.cwd()
    for base in (cwd, cwd.parent, cwd.parent.parent):
        b = base / "backend"
        if (b / "eval_quiz_t5.py").exists():
            return b.resolve()
    raise FileNotFoundError(
        "Could not find backend/ — open this notebook from Educonnect or notebooks and run again."
    )


BACKEND = find_backend()
sys.path.insert(0, str(BACKEND))
DATA_DIR = BACKEND / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

from eval_quiz_t5 import (
    compute_bleu,
    compute_rouge,
    compute_meteor,
    plot_metrics,
    plot_compare,
)

MODEL_NAME = "google/flan-t5-base"
NOTEBOOK_MODEL_DIR = BACKEND / "models" / "notebook_flant5_qg"
print("BACKEND =", BACKEND)
print("Model will be saved to:", NOTEBOOK_MODEL_DIR)

BACKEND = C:\Users\USER\OneDrive\Desktop\The Actual Educonnect (2)\AI-Enabled-peer-to-peer-learning-framework\The Actual Educonnect\Educonnect\backend
Model will be saved to: C:\Users\USER\OneDrive\Desktop\The Actual Educonnect (2)\AI-Enabled-peer-to-peer-learning-framework\The Actual Educonnect\Educonnect\backend\models\notebook_flant5_qg


## 2. Load QG data and the same split as Notebook A

**What this section does:** Loads `learning_resources_qg.json`, filters rows where the answer appears in the context, then either reads **`qg_split_manifest.json`** (preferred, produced by Notebook A) or rebuilds the same 70/15/15 split with seed `42`. Training uses **train** indices; evaluation uses **test** indices so metrics align with the baseline notebook.

In [2]:
qg_path = DATA_DIR / "learning_resources_qg.json"
if not qg_path.exists():
    from build_learning_resources_qg import build_qg_pairs
    from load_learning_resources import get_learning_resources

    resources = get_learning_resources(enrich_quiz_source=True)
    pairs = build_qg_pairs(resources, max_per_resource=5)
    raw = [
        {"context": c, "answer": a, "question": q, "category": cat}
        for c, a, q, cat in pairs
    ]
    qg_path.write_text(json.dumps(raw, indent=2), encoding="utf-8")
else:
    raw = json.loads(qg_path.read_text(encoding="utf-8"))

valid_data = [it for it in raw if it.get("answer") and it["answer"] in it["context"]]

manifest_path = DATA_DIR / "qg_split_manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    train_ix = manifest["train"]
    val_ix = manifest["val"]
    test_ix = manifest["test"]
    print("Loaded split from", manifest_path)
else:
    SPLIT_SEED = 42
    rng = random.Random(SPLIT_SEED)
    n = len(valid_data)
    indices = list(range(n))
    rng.shuffle(indices)
    n_train = int(0.70 * n)
    n_val = int(0.15 * n)
    train_ix = indices[:n_train]
    val_ix = indices[n_train : n_train + n_val]
    test_ix = indices[n_train + n_val :]
    manifest_path.write_text(
        json.dumps(
            {
                "seed": SPLIT_SEED,
                "n_valid": n,
                "train": train_ix,
                "val": val_ix,
                "test": test_ix,
            }
        ),
        encoding="utf-8",
    )
    print("Created split (run Notebook A first next time for a shared manifest).")

train_items = [valid_data[i] for i in train_ix]
val_items = [valid_data[i] for i in val_ix]
test_items = [valid_data[i] for i in test_ix]
print("Train:", len(train_items), " Val:", len(val_items), " Test:", len(test_items))

Loaded split from C:\Users\USER\OneDrive\Desktop\The Actual Educonnect (2)\AI-Enabled-peer-to-peer-learning-framework\The Actual Educonnect\Educonnect\backend\data\qg_split_manifest.json
Train: 200  Val: 43  Test: 44


## 3. Build training and validation sequences (highlighted context → question)

**What this section does:** Converts each **train** and **validation** row into a Flan-T5 input string by replacing the first occurrence of the answer in the context with `<hl> answer </hl>`, prefixed with `generate question: `. Targets are the reference questions. Rows where replacement is impossible are skipped. The validation tensors are used each epoch during fine-tuning and for optional generation metrics later.

In [3]:
def row_to_input_target(item):
    ctx, ans, q = item["context"], item["answer"], item["question"]
    if ans not in ctx:
        return None
    highlighted = ctx.replace(ans, f"<hl> {ans} </hl>", 1)
    inp = f"generate question: {highlighted}"
    return inp, q


train_inputs, train_targets = [], []
for item in train_items:
    out = row_to_input_target(item)
    if out:
        train_inputs.append(out[0])
        train_targets.append(out[1])

val_inputs, val_targets = [], []
for item in val_items:
    out = row_to_input_target(item)
    if out:
        val_inputs.append(out[0])
        val_targets.append(out[1])

print("Training sequences:", len(train_inputs), "  Validation sequences:", len(val_inputs))

Training sequences: 200   Validation sequences: 43


## 4. Fine-tune Flan-T5

**What this section does:** Tokenizes inputs and targets, runs Hugging Face `Trainer`, and saves weights under `backend/models/notebook_flant5_qg/`.

**Default training:** The fine-tune cell runs **3 full epochs** on all training rows (`DRY_RUN = False`), then **automatically** runs evaluation (Section 5) and baseline comparison (Section 6) so you do not have to run those cells manually.

**Smoke test:** Set `DRY_RUN = True` in that cell for 1 epoch and 100 rows only (if you hit OOM or want a quick check).

**GPU:** Run the **GPU check** cell; `CUDA available: True` means `Trainer` will use the GPU. If VRAM is tight, keep `DRY_RUN = False` but lower `BATCH_SIZE` in the `else` branch of the training cell.

In [4]:
# Step 2: confirm GPU (PyTorch must report CUDA for GPU training)
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA (PyTorch build):", torch.version.cuda)
else:
    print("Using CPU — training is slower. Install a CUDA-enabled PyTorch build for your GPU driver if you expect GPU training.")

CUDA available: False
Using CPU — training is slower. Install a CUDA-enabled PyTorch build for your GPU driver if you expect GPU training.


In [6]:
%pip install accelerate transformers
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    T5ForConditionalGeneration,
    Trainer,
    TrainingArguments,
)

# Built in Section 3 — you must run cells 1→3 (load data + build sequences) before this cell.
_required = ("train_inputs", "train_targets", "val_inputs", "val_targets", "MODEL_NAME")
_missing = [n for n in _required if n not in globals()]
if _missing:
    raise RuntimeError(
        "Not ready to train: missing "
        + ", ".join(_missing)
        + ". Run all cells above in order (Sections 2–3) so train_inputs / val_inputs exist, then run this cell."
    )

# Default False: 3 epochs on up to MAX_TRAIN rows (else branch), then eval + baseline compare.
# Set True only for a quick smoke test (1 epoch, 100 rows).
DRY_RUN = False

if DRY_RUN:
    EPOCHS = 1
    BATCH_SIZE = 1
    MAX_TRAIN = 100
    INPUT_MAX_LEN = 256
    TARGET_MAX_LEN = 48
    USE_FP16 = False
else:
    EPOCHS = 3
    BATCH_SIZE = 2 if torch.cuda.is_available() else 1
    MAX_TRAIN = 2000  # cap training rows; slice is [:2000] so smaller sets use all rows. Use None for no cap.
    INPUT_MAX_LEN = 512
    TARGET_MAX_LEN = 64
    USE_FP16 = bool(torch.cuda.is_available())

if MAX_TRAIN:
    train_inputs = train_inputs[:MAX_TRAIN]
    train_targets = train_targets[:MAX_TRAIN]

print(
    "Training config:",
    {
        "DRY_RUN": DRY_RUN,
        "EPOCHS": EPOCHS,
        "BATCH_SIZE": BATCH_SIZE,
        "train_rows": len(train_inputs),
        "INPUT_MAX_LEN": INPUT_MAX_LEN,
        "TARGET_MAX_LEN": TARGET_MAX_LEN,
        "USE_FP16": USE_FP16,
        "cuda": torch.cuda.is_available(),
    },
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
pad_id = tokenizer.pad_token_id or 0


def tokenize_fn(examples):
    model_inputs = tokenizer(
        examples["input"],
        max_length=INPUT_MAX_LEN,
        truncation=True,
        padding="max_length",
    )
    labels = tokenizer(
        examples["target"],
        max_length=TARGET_MAX_LEN,
        truncation=True,
        padding="max_length",
    )
    model_inputs["labels"] = [
        [(l if l != pad_id else -100) for l in label] for label in labels["input_ids"]
    ]
    return model_inputs


ds = Dataset.from_dict({"input": train_inputs, "target": train_targets})
tokenized_train = ds.map(tokenize_fn, batched=True, remove_columns=ds.column_names)

val_ds = Dataset.from_dict({"input": val_inputs, "target": val_targets})
tokenized_val = val_ds.map(tokenize_fn, batched=True, remove_columns=val_ds.column_names)

NOTEBOOK_MODEL_DIR.mkdir(parents=True, exist_ok=True)
training_args = TrainingArguments(
    output_dir=str(NOTEBOOK_MODEL_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_steps=min(500, max(1, len(train_inputs) // 10)),
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=USE_FP16,
    dataloader_pin_memory=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
)
trainer.train()
trainer.save_model(str(NOTEBOOK_MODEL_DIR))
tokenizer.save_pretrained(str(NOTEBOOK_MODEL_DIR))
print("Saved:", NOTEBOOK_MODEL_DIR)

# --- Auto-run Sections 5–6 (eval + baseline compare) after all epochs finish ---
import subprocess

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "protobuf", "sentencepiece"],
    check=False,
)


def generate_questions(model, tokenizer, items, batch_size=8):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    contexts = [it["context"] for it in items]
    answers = [it["answer"] for it in items]
    refs = [it["question"] for it in items]
    preds = []
    for i in range(0, len(contexts), batch_size):
        batch_ctx = contexts[i : i + batch_size]
        batch_ans = answers[i : i + batch_size]
        inputs = [c.replace(a, f"<hl> {a} </hl>", 1) for c, a in zip(batch_ctx, batch_ans)]
        inputs = [f"generate question: {inp}" for inp in inputs]
        enc = tokenizer(inputs, return_tensors="pt", padding=True, truncation=True, max_length=512)
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = model.generate(**enc, max_length=64, num_beams=4)
        preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    preds = [p.strip() for p in preds]
    return preds, refs


try:
    tok_eval = AutoTokenizer.from_pretrained(str(NOTEBOOK_MODEL_DIR), use_fast=False)
except Exception:
    tok_eval = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
    tok_eval.save_pretrained(str(NOTEBOOK_MODEL_DIR))

mdl_eval = T5ForConditionalGeneration.from_pretrained(str(NOTEBOOK_MODEL_DIR))
mdl_eval.eval()

vp, vr = generate_questions(mdl_eval, tok_eval, val_items)
ft_val_metrics = {
    "BLEU": compute_bleu(vp, vr),
    "ROUGE-L": compute_rouge(vp, vr)["rougeL_fmeasure"],
    "METEOR": compute_meteor(vp, vr),
}
print("--- Validation ---")
for k, v in ft_val_metrics.items():
    print(f"{k:12} {v:.4f}")
(DATA_DIR / "ft_val_metrics.json").write_text(json.dumps(ft_val_metrics, indent=2), encoding="utf-8")
plot_metrics(
    ft_val_metrics,
    DATA_DIR / "ft_val_eval_results.png",
    title="Fine-tuned Flan-T5 QG — validation set",
)

tp, tr = generate_questions(mdl_eval, tok_eval, test_items)
ft_metrics = {
    "BLEU": compute_bleu(tp, tr),
    "ROUGE-L": compute_rouge(tp, tr)["rougeL_fmeasure"],
    "METEOR": compute_meteor(tp, tr),
}
print("--- Test ---")
for k, v in ft_metrics.items():
    print(f"{k:12} {v:.4f}")
(DATA_DIR / "ft_metrics.json").write_text(json.dumps(ft_metrics, indent=2), encoding="utf-8")
plot_metrics(
    ft_metrics,
    DATA_DIR / "ft_eval_results.png",
    title="Fine-tuned Flan-T5 QG — test set",
)

baseline_path = DATA_DIR / "baseline_metrics.json"
if baseline_path.exists():
    baseline_metrics = json.loads(baseline_path.read_text(encoding="utf-8"))
    print("Baseline:", baseline_metrics)
    print("Fine-tuned:", ft_metrics)
    plot_compare(baseline_metrics, ft_metrics, DATA_DIR / "baseline_vs_ft_compare.png")
else:
    print(
        "No baseline_metrics.json — run Notebook A first for baseline_vs_ft_compare.png."
    )

print("Done: all epochs + eval + baseline comparison (if baseline_metrics.json exists).")

Note: you may need to restart the kernel to use updated packages.
Training config: {'DRY_RUN': False, 'EPOCHS': 3, 'BATCH_SIZE': 1, 'train_rows': 200, 'INPUT_MAX_LEN': 512, 'TARGET_MAX_LEN': 64, 'USE_FP16': False, 'cuda': False}


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/43 [00:00<?, ? examples/s]

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss


: 

In [1]:
%pip install -q protobuf sentencepiece

Note: you may need to restart the kernel to use updated packages.


## 5. Evaluate on the test split (same examples as Notebook A)

**Usually already done:** Section 4 runs this automatically after all training epochs finish.

**Re-run the code cell below** only if you did not run Section 4’s full cell, changed the model folder, or want fresh metrics without retraining. It loads the checkpoint, generates questions with beam search, computes BLEU / ROUGE-L / METEOR, and saves `ft_eval_results.png` and `ft_metrics.json`.

In [ ]:
import torch
from transformers import T5ForConditionalGeneration, AutoTokenizer

if "NOTEBOOK_MODEL_DIR" not in globals() or "val_items" not in globals():
    raise RuntimeError(
        "Run Sections 1–3 first (NOTEBOOK_MODEL_DIR, val_items, test_items), or run Section 4 which includes eval."
    )


def generate_questions(model, tokenizer, items, batch_size=8):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    contexts = [it["context"] for it in items]
    answers = [it["answer"] for it in items]
    refs = [it["question"] for it in items]
    preds = []
    for i in range(0, len(contexts), batch_size):
        batch_ctx = contexts[i : i + batch_size]
        batch_ans = answers[i : i + batch_size]
        inputs = [c.replace(a, f"<hl> {a} </hl>", 1) for c, a in zip(batch_ctx, batch_ans)]
        inputs = [f"generate question: {inp}" for inp in inputs]
        enc = tokenizer(inputs, return_tensors="pt", padding=True, truncation=True, max_length=512)
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = model.generate(**enc, max_length=64, num_beams=4)
        preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    preds = [p.strip() for p in preds]
    return preds, refs


try:
    tok = AutoTokenizer.from_pretrained(str(NOTEBOOK_MODEL_DIR), use_fast=False)
except Exception:
    tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
    tok.save_pretrained(str(NOTEBOOK_MODEL_DIR))

mdl = T5ForConditionalGeneration.from_pretrained(str(NOTEBOOK_MODEL_DIR))
mdl.eval()

# Validation
vp, vr = generate_questions(mdl, tok, val_items)
ft_val_metrics = {
    "BLEU": compute_bleu(vp, vr),
    "ROUGE-L": compute_rouge(vp, vr)["rougeL_fmeasure"],
    "METEOR": compute_meteor(vp, vr),
}
print("--- Validation ---")
for k, v in ft_val_metrics.items():
    print(f"{k:12} {v:.4f}")
(DATA_DIR / "ft_val_metrics.json").write_text(json.dumps(ft_val_metrics, indent=2), encoding="utf-8")
plot_metrics(
    ft_val_metrics,
    DATA_DIR / "ft_val_eval_results.png",
    title="Fine-tuned Flan-T5 QG — validation set",
)

# Test (compare with Notebook A baseline on this split)
tp, tr = generate_questions(mdl, tok, test_items)
ft_metrics = {
    "BLEU": compute_bleu(tp, tr),
    "ROUGE-L": compute_rouge(tp, tr)["rougeL_fmeasure"],
    "METEOR": compute_meteor(tp, tr),
}
print("--- Test ---")
for k, v in ft_metrics.items():
    print(f"{k:12} {v:.4f}")
(DATA_DIR / "ft_metrics.json").write_text(json.dumps(ft_metrics, indent=2), encoding="utf-8")
plot_metrics(
    ft_metrics,
    DATA_DIR / "ft_eval_results.png",
    title="Fine-tuned Flan-T5 QG — test set",
)

NameError: name 'NOTEBOOK_MODEL_DIR' is not defined

## 6. Comparison: baseline (Notebook A) vs fine-tuned model

**Usually already done:** Section 4 also runs this comparison when `baseline_metrics.json` exists.

**Re-run the code cell below** to refresh the chart after re-evaluating. Loads `baseline_metrics.json` from Notebook A, prints baseline vs fine-tuned metrics, and writes `baseline_vs_ft_compare.png`. If the file is missing, run Notebook A first (same test split).

In [ ]:
baseline_path = DATA_DIR / "baseline_metrics.json"
if baseline_path.exists():
    baseline_metrics = json.loads(baseline_path.read_text(encoding="utf-8"))
    print("Baseline:", baseline_metrics)
    print("Fine-tuned:", ft_metrics)
    plot_compare(baseline_metrics, ft_metrics, DATA_DIR / "baseline_vs_ft_compare.png")
else:
    print(
        "No baseline_metrics.json — run Notebook A first to save baseline scores, then re-run this cell."
    )